# Scenario Evaluation — Manual Reassignment Tool

Explore the impact of manual labor→driver reassignments on schedule KPIs.

**Section 1 — Scenario Editor**  
Load a base solution, use the interactive panel to reassign labors to different drivers,
preview the impact, and save the result as a named scenario.

**Section 2 — Scenario Comparison**  
Compare two or more saved scenarios side-by-side using KPI tables, Gantt charts,
distance figures, and route maps.

> Both sections save to and read from the **same `scenarios/` directory** — no extra path configuration needed.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
from typing import List, Optional, Tuple

import pandas as pd
from IPython.display import display

# ── Project root on sys.path ────────────────────────────────────────────────
_PROJECT_ROOT = Path("../..").resolve()
if str(_PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / "src"))

# ── Scenario editor helpers ─────────────────────────────────────────────────
from alfred.analysis.scenario_editor import (
    ScenarioSet,
    apply_reassignments,
    build_labor_editor_table,
    build_multi_scenario_overview_table,
    build_service_display_info,
    load_driver_names,
    load_scenario_base,
    load_scenarios_for_comparison,
    reconstruct_scenario,
    save_scenario,
)

# ── Existing analysis utilities ─────────────────────────────────────────────
from alfred.analysis.compare_solutions import build_overview_table, load_and_prepare
from alfred.analysis.solution_evaluation import (
    build_driver_distance_figure,
    build_gantt_figure,
    build_route_map,
    build_service_distance_figure,
    compute_payload_summary,
)
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

print("Imports OK — project root:", _PROJECT_ROOT)

Imports OK — project root: /Users/jbeta/Documents/AlfredProject/AlfredDEV


---
## Configuration

Set the paths to the experiment folder once here. Both sections will use the same `scenarios/` directory inside it.

In [2]:
# ── Paths — edit this cell ──────────────────────────────────────────────────
# Root of the experiment run (must contain output/ and api_snapshot/ sub-folders)
EXPERIMENT_DIR: Path = (
    _PROJECT_ROOT / "misc" / "experiments" / "scenarios" / "20260508"
)

# Derived paths (no edits needed below this line)
BASE_PAYLOAD:     Path          = EXPERIMENT_DIR / "output" / "output_payload.json"
DRIVER_DIRECTORY: Path          = EXPERIMENT_DIR / "api_snapshot" / "driver_directory_snapshot.json"
INPUT_FILE:       Optional[Path] = EXPERIMENT_DIR / "api_snapshot" / "optimization_input_snapshot.json"

# All scenarios (base copy + manual edits) land here
SCENARIOS_DIR: Path = EXPERIMENT_DIR / "scenarios"

# Filter to a single planning date (YYYY-MM-DD), or None to load all
PLANNING_DATE: Optional[str] = None

# Distance computation method — 'osrm' (default) or 'haversine'
DISTANCE_METHOD: str = DEFAULT_DISTANCE_METHOD

print(f"Experiment  : {EXPERIMENT_DIR}")
print(f"Base payload: {BASE_PAYLOAD}")
print(f"Scenarios   : {SCENARIOS_DIR}")
print(f"Driver dir  : {DRIVER_DIRECTORY} — exists: {DRIVER_DIRECTORY.exists()}")
print(f"Input file  : {INPUT_FILE} — exists: {INPUT_FILE.exists() if INPUT_FILE else False}")

Experiment  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508
Base payload: /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/output/output_payload.json
Scenarios   : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/scenarios
Driver dir  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/api_snapshot/driver_directory_snapshot.json — exists: True
Input file  : /Users/jbeta/Documents/AlfredProject/AlfredDEV/misc/experiments/scenarios/20260508/api_snapshot/optimization_input_snapshot.json — exists: True


---
## Section 1 — Scenario Editor

Run the cells in order:
1. **Load** the base solution
2. **Reassign** labors using the interactive panel
3. **Save** the result as a new scenario

In [3]:
# ── Load base solution ───────────────────────────────────────────────────────
_base = load_scenario_base(
    payload_path=BASE_PAYLOAD,
    driver_directory_path=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    input_file=INPUT_FILE,
)

# Load human-readable metadata for the widget UI
_driver_names = load_driver_names(DRIVER_DIRECTORY)   # {id_str: "First Last"}
_service_info = build_service_display_info(INPUT_FILE) # {labor_id: {labor_name, from_address, …}}

# Build the editor DataFrame (source of truth for the widget)
_editor_df = build_labor_editor_table(_base)

address_crosswalk_not_found path=/Users/jbeta/Documents/AlfredProject/AlfredDEV/data/examples/raw_files/address.csv


[driver_home_lookup] 12 drivers loaded
[coord_lookup] 26 labors | method='osrm'
[load_scenario_base] 21 labors | 9 drivers | 45 segments


In [4]:
# ── Interactive Reassignment Panel ───────────────────────────────────────────
import ipywidgets as W
from IPython.display import HTML, clear_output

# ── Helpers ──────────────────────────────────────────────────────────────────

def _driver_label(driver_id) -> str:
    """Format a driver for dropdown display: 'First Last (ID)'."""
    if driver_id is None:
        return "— unassigned —"
    name = _driver_names.get(str(driver_id), "")
    return f"{name}  ({driver_id})" if name else str(driver_id)

def _labor_header(row) -> str:
    """One-line summary for a labor row."""
    info = _service_info.get(row["labor_id"]) or _service_info.get(str(row["labor_id"])) or {}
    name = info.get("labor_name") or row["labor_type"] or "Labor"
    start = (row["actual_start"] or "")[:16].replace("T", "  ")
    return f"{name}  ·  {start}"

def _labor_subtext(row) -> str:
    """From → To address text for the labor card."""
    info = _service_info.get(row["labor_id"]) or _service_info.get(str(row["labor_id"])) or {}
    frm = info.get("from_address", "")
    to  = info.get("to_address", "")
    if frm or to:
        return f"{frm}  →  {to}"
    return f"Service {row['service_id']}"

# ── Build sorted driver option list ─────────────────────────────────────────
_all_driver_ids = sorted(
    {str(r["driver_id"]) for r in _base.rows if r["driver_id"] is not None}
    | set(_driver_names.keys()),
    key=lambda d: _driver_names.get(d, d),
)
_driver_options = [(f"{_driver_names.get(d, d)}  ({d})", d) for d in _all_driver_ids]

# ── State: one Dropdown per labor row ────────────────────────────────────────
_dropdowns: dict = {}   # labor_id → Dropdown widget

# ── CSS ──────────────────────────────────────────────────────────────────────
_CSS = """
<style>
.sc-panel { font-family: 'Inter', sans-serif; max-width: 900px; }
.sc-card {
    border: 1px solid #e2e8f0;
    border-radius: 8px;
    padding: 10px 14px;
    margin-bottom: 8px;
    background: #fff;
    display: flex;
    align-items: flex-start;
    gap: 14px;
}
.sc-card.sc-changed { border-left: 4px solid #f59e0b; background: #fffbeb; }
.sc-card-body { flex: 1; min-width: 0; }
.sc-card-title { font-weight: 600; font-size: 13px; color: #1e293b; margin-bottom: 2px; }
.sc-card-sub   { font-size: 11px; color: #64748b; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }
.sc-badge {
    display: inline-block; padding: 1px 7px; border-radius: 99px;
    font-size: 10px; font-weight: 600; letter-spacing: .4px; margin-top: 4px;
}
.sc-badge-vt   { background:#dbeafe; color:#1d4ed8; }
.sc-badge-shop { background:#f0fdf4; color:#15803d; }
.sc-badge-warn { background:#fef3c7; color:#b45309; }
.sc-section-header {
    font-size: 11px; font-weight: 700; text-transform: uppercase;
    letter-spacing: .8px; color: #94a3b8; margin: 14px 0 6px;
}
.sc-summary {
    background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 8px;
    padding: 10px 16px; margin-top: 10px; font-size: 12px;
}
.sc-summary b { color: #1e293b; }
.sc-infeasible { color: #dc2626; font-weight: 600; }
</style>
"""

# ── Build the card list ───────────────────────────────────────────────────────

def _labor_type_badge(labor_type: str) -> str:
    vt_types = {"alfred_initial_transport", "alfred_transport"}
    if labor_type in vt_types:
        return '<span class="sc-badge sc-badge-vt">transport</span>'
    return '<span class="sc-badge sc-badge-shop">shop</span>'

# Group rows by service for section headers
_rows_by_service: dict = {}
for _, row in _editor_df.iterrows():
    _rows_by_service.setdefault(row["service_id"], []).append(row)

_card_widgets = []   # ordered list of (HTML-label-widget, Dropdown)

for svc_id, svc_rows in _rows_by_service.items():
    # Section header for the service
    info0 = (_service_info.get(svc_rows[0]["labor_id"]) or
             _service_info.get(str(svc_rows[0]["labor_id"])) or {})
    frm = info0.get("from_address", "")
    to  = info0.get("to_address", "")
    route_text = f"{frm}  →  {to}" if (frm or to) else f"Service {svc_id}"
    svc_header = W.HTML(
        f'<div class="sc-section-header" style="margin-top:10px">'
        f'Service {svc_id} &nbsp;·&nbsp; <span style="font-weight:400;text-transform:none">'
        f'{route_text}</span></div>'
    )
    _card_widgets.append(svc_header)

    for _, row in [(i, r) for i, r in enumerate(svc_rows)]:
        labor_id     = row["labor_id"]
        orig_driver  = str(row["original_driver"]) if row["original_driver"] is not None else None
        labor_type   = row["labor_type"] or ""
        header_text  = _labor_header(row)
        badge        = _labor_type_badge(labor_type)

        label_html = W.HTML(
            f'<div class="sc-card-body">'
            f'  <div class="sc-card-title">{header_text}</div>'
            f'  <div class="sc-card-sub">{_labor_subtext(row)}</div>'
            f'  {badge}'
            f'</div>'
        )

        # Dropdown: current driver pre-selected
        dd = W.Dropdown(
            options=_driver_options,
            value=orig_driver,
            layout=W.Layout(width="260px", min_width="200px"),
        )
        _dropdowns[labor_id] = dd

        card = W.HBox(
            [label_html, dd],
            layout=W.Layout(
                border="1px solid #e2e8f0",
                border_radius="8px",
                padding="8px 12px",
                margin="0 0 6px 0",
                background="white",
                align_items="center",
            ),
        )
        _card_widgets.append(card)

# ── Scenario name input + Generate button ────────────────────────────────────
_scenario_name_input = W.Text(
    value="scenario_v1",
    placeholder="e.g. scenario_v1",
    description="Scenario name:",
    style={"description_width": "120px"},
    layout=W.Layout(width="380px"),
)

_generate_btn = W.Button(
    description=" Generate & Save Scenario",
    button_style="success",
    icon="check",
    layout=W.Layout(width="240px", height="36px"),
    tooltip="Reconstruct the schedule with your reassignments and save to the scenarios folder",
)

_result_output = W.Output()

# Storage for the last generated scenario (used by Section 2 without re-running)
_last_scenario: dict = {"rows": None, "segments": None, "label": None, "path": None}


def _on_generate(btn):
    """Read dropdowns, reconstruct, and save."""
    with _result_output:
        clear_output(wait=True)

        # Collect reassignments
        _orig_map = dict(zip(_editor_df["labor_id"], _editor_df["original_driver"]))
        reassignments = {}
        for lid, dd in _dropdowns.items():
            orig = str(_orig_map.get(lid)) if _orig_map.get(lid) is not None else None
            new  = dd.value
            if new != orig:
                reassignments[lid] = new

        display(HTML(f"<p style='color:#64748b;font-size:12px'>Applying <b>{len(reassignments)}</b> reassignment(s)…</p>"))

        new_rows, warnings = apply_reassignments(_base.rows, reassignments)

        if warnings:
            warn_html = "<br>".join(f"⚠ {w}" for w in warnings)
            display(HTML(f"<div class='sc-badge sc-badge-warn' style='padding:6px 10px;margin:4px 0;font-size:11px'>{warn_html}</div>"))

        new_rows, new_segs = reconstruct_scenario(
            new_rows,
            _base.points_lookup,
            _base.driver_home_lookup,
            _base.params,
            DISTANCE_METHOD,
        )

        # Report infeasibilities
        infeasible = [r for r in new_rows if r.get("is_infeasible")]
        if infeasible:
            rows_html = "".join(
                f"<tr><td>{r['labor_id']}</td><td>{r['service_id']}</td>"
                f"<td>{_driver_label(r['driver_id'])}</td>"
                f"<td>{str(r['actual_start'])[:16]}</td></tr>"
                for r in infeasible
            )
            display(HTML(
                f"<div class='sc-summary'>"
                f"<span class='sc-infeasible'>⚠ {len(infeasible)} labor(s) infeasible after reassignment:</span>"
                f"<table style='margin-top:6px;font-size:11px;border-collapse:collapse;width:100%'>"
                f"<thead><tr style='color:#94a3b8'><th>Labor</th><th>Service</th><th>Driver</th><th>Start</th></tr></thead>"
                f"<tbody>{rows_html}</tbody></table></div>"
            ))
        else:
            display(HTML("<p style='color:#16a34a;font-size:12px'>✓ No infeasible labors detected.</p>"))

        # Save
        label = _scenario_name_input.value.strip() or "scenario_v1"
        SCENARIOS_DIR.mkdir(parents=True, exist_ok=True)
        out_path = SCENARIOS_DIR / f"{label}.json"
        saved = save_scenario(_base.services, new_rows, out_path, label)

        # Cache for Section 2
        _last_scenario["rows"]     = new_rows
        _last_scenario["segments"] = new_segs
        _last_scenario["label"]    = label
        _last_scenario["path"]     = saved

        # Summary table
        summary = compute_payload_summary(
            new_rows,
            tiempo_gracia_min=_base.params.tiempo_gracia_min,
            segments=new_segs,
        )
        kpi_rows = [
            ("Labors assigned",      summary.get("labors_assigned", "—")),
            ("Infeasible",           summary.get("labors_infeasible", "—")),
            ("Total labor dist (km)",f"{summary.get('total_labor_distance_km', 0):.1f}"),
            ("Total move dist (km)", f"{summary.get('total_driver_move_distance_km', 0):.1f}"),
        ]
        kpi_html = "".join(f"<tr><td style='padding:2px 10px 2px 0;color:#64748b'>{k}</td><td style='font-weight:600'>{v}</td></tr>" for k, v in kpi_rows)
        display(HTML(
            f"<div class='sc-summary'>"
            f"<b>Scenario saved</b> → <code>{saved}</code>"
            f"<table style='margin-top:8px;font-size:12px'>{kpi_html}</table>"
            f"</div>"
        ))

        # Gantt preview
        new_drivers = sorted({r["driver_id"] for r in new_rows if r["driver_id"] is not None})
        build_gantt_figure(new_segs, new_drivers, f"{label} — preview").show()


_generate_btn.on_click(_on_generate)

# ── Render the full panel ─────────────────────────────────────────────────────
display(HTML(_CSS))
display(W.VBox(
    [
        W.HTML('<div class="sc-panel">'),
        W.HTML('<div class="sc-section-header">Service assignments — edit the dropdowns then click Generate</div>'),
        *_card_widgets,
        W.HTML('<hr style="margin:14px 0;border-color:#e2e8f0">'),
        W.HBox([_scenario_name_input, _generate_btn], layout=W.Layout(gap="12px", align_items="center")),
        _result_output,
        W.HTML('</div>'),
    ],
    layout=W.Layout(max_width="920px"),
))

---
## Section 2 — Scenario Comparison

All saved scenarios (including the base) are read from `SCENARIOS_DIR`.
The base payload is automatically copied there on first run so it can be
compared on equal footing with any edited scenario.

- The **first entry** in `SCENARIO_PATHS` is the baseline for Δ columns.
- Run this section independently by pointing `SCENARIOS_DIR` to any folder
  containing pre-saved scenario JSON files.

In [5]:
# ── Copy the base payload into the scenarios folder (once) ───────────────────
import shutil

SCENARIOS_DIR.mkdir(parents=True, exist_ok=True)
_base_copy = SCENARIOS_DIR / "base.json"
if not _base_copy.exists():
    shutil.copy2(BASE_PAYLOAD, _base_copy)
    print(f"Base payload copied → {_base_copy}")
else:
    print(f"Base already present: {_base_copy}")

# ── List all available scenarios in the folder ───────────────────────────────
_available = sorted(SCENARIOS_DIR.glob("*.json"))
print(f"\nAvailable scenarios in {SCENARIOS_DIR}:")
for p in _available:
    print(f"  {p.name}")

In [6]:
# ── Section 2 Configuration ──────────────────────────────────────────────────
# Scenarios to compare — (path, label) list; first entry = baseline.
# By default: compare base vs. the last scenario generated in Section 1
# (or list explicit paths from SCENARIOS_DIR if running Section 2 standalone).
SCENARIO_PATHS: List[Tuple[Path, str]] = [
    (SCENARIOS_DIR / "base.json",        "base"),
    # Add more:
    (SCENARIOS_DIR / "scenario_v1.json", "scenario_v1"),
    # (SCENARIOS_DIR / "scenario_v2.json", "scenario_v2"),
]

# If Section 1 was just run, append the new scenario automatically
if _last_scenario.get("path") and _last_scenario["path"] not in [p for p, _ in SCENARIO_PATHS]:
    SCENARIO_PATHS.append((_last_scenario["path"], _last_scenario["label"]))
    print(f"Auto-added last scenario: {_last_scenario['label']}")

print(f"Comparing: {[lbl for _, lbl in SCENARIO_PATHS]}")

In [7]:
# ── Load all scenarios ───────────────────────────────────────────────────────
_scenario_set = load_scenarios_for_comparison(
    scenario_paths=SCENARIO_PATHS,
    driver_directory_path=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    input_file=INPUT_FILE,
)

In [8]:
# ── KPI Overview Table (N scenarios) ────────────────────────────────────────
_overview = build_multi_scenario_overview_table(
    _scenario_set,
    baseline_label=SCENARIO_PATHS[0][1],
)
display(
    _overview.style
    .format(precision=2, na_rep="—")
    .set_caption(f"Scenario Comparison — Δ vs '{SCENARIO_PATHS[0][1]}'")
)

In [ ]:
# ── Pairwise comparison — runs only for exactly 2 scenarios ─────────────────
if len(SCENARIO_PATHS) == 2:
    _la, _lb = SCENARIO_PATHS[0][1], SCENARIO_PATHS[1][1]
    _pair = load_and_prepare(
        sol_a=SCENARIO_PATHS[0][0],
        sol_b=SCENARIO_PATHS[1][0],
        driver_directory=DRIVER_DIRECTORY,
        planning_date=PLANNING_DATE,
        distance_method=DISTANCE_METHOD,
        input_file=INPUT_FILE,
    )
    display(build_overview_table(_pair, _la, _lb))
else:
    print(f"Pairwise cell skipped ({len(SCENARIO_PATHS)} scenarios loaded).")

In [ ]:
# ── Gantt Charts — one per scenario ─────────────────────────────────────────
for label, rows, segs in zip(
    _scenario_set.labels,
    _scenario_set.rows_list,
    _scenario_set.segments_list,
):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_gantt_figure(segs, _drivers, label).show()

In [13]:
# ── Distance per Service — one chart per scenario ────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    build_service_distance_figure(rows, _scenario_set.all_services, label).show()

In [14]:
# ── Distance per Driver — one chart per scenario ─────────────────────────────
for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
    _drivers = sorted({r["driver_id"] for r in rows if r["driver_id"] is not None})
    build_driver_distance_figure(rows, _drivers, label).show()

In [13]:
# ── Route Maps — driver dropdown, one map per scenario ───────────────────────
import ipywidgets as W
from IPython.display import HTML

_drv_options = [
    (f"{_driver_names.get(d, d)}  ({d})", d)
    for d in _scenario_set.all_drivers
]
_drv_dropdown = W.Dropdown(
    options=_drv_options,
    description="Driver:",
    layout=W.Layout(width="320px"),
)
_map_out = W.Output()


def _render_maps(change):
    _map_out.clear_output(wait=True)
    did = change["new"]
    with _map_out:
        for label, rows in zip(_scenario_set.labels, _scenario_set.rows_list):
            if not any(r["driver_id"] == str(did) for r in rows):
                display(W.HTML(f"<p style='color:#94a3b8;font-size:12px'><b>{label}</b>: driver {did!r} has no labors.</p>"))
                continue
            home_wkt = (_scenario_set.driver_home_lookup or {}).get(str(did))
            display(W.HTML(f"<h4 style='margin:12px 0 4px'>{label}</h4>"))
            try:
                fig = build_route_map(
                    services=None,
                    rows=rows,
                    driver_id=did,
                    driver_home_wkt=home_wkt,
                    label=label,
                    points_lookup=_scenario_set.points_lookup,
                )
                display(HTML(fig._repr_html_()))
            except Exception as exc:
                display(W.HTML(f"<p style='color:red'>Map error: {exc}</p>"))


_drv_dropdown.observe(_render_maps, names="value")
_render_maps({"new": _drv_dropdown.value})
display(_drv_dropdown, _map_out)

Dropdown(description='Driver:', layout=Layout(width='320px'), options=(('Ivan Dario Pinta  (10451)', '10451'),…

Output()